# APUSH FRQ Grader v3
Fresh Colab workflow for v3. It clones the repository, keeps run outputs on Drive, defaults to set1, and never opens set2 without an explicit locked-final flag.

## 1. Clone/update the repository and install dependencies

In [44]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/aryanjverma/slm.git'
REPO = Path('/content/apush-frq-grader-slm')
if (REPO / '.git').exists():
    subprocess.run(['git', 'pull', '--ff-only', 'origin', 'main'], cwd=REPO, check=True)
elif REPO.exists():
    raise RuntimeError(f'{REPO} exists but is not a git checkout; remove or rename it first')
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
print(f'Repository: {REPO}\nCommit: {COMMIT}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[train]'], check=True)

def run_streaming(command):
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    return_code = process.wait()
    if return_code:
        raise RuntimeError(f'Command failed with exit code {return_code}: {command}')


Repository: /content/apush-frq-grader-slm
Commit: 869eee44da242951b73ec1f85e09d386540c22f3


## 2. Mount Drive and configure immutable run outputs

In [29]:
from google.colab import drive
drive.mount('/content/drive')
RUN_ROOT = Path('/content/drive/MyDrive/slm_v3')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Run outputs: {RUN_ROOT}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Run outputs: /content/drive/MyDrive/slm_v3


## 3. Reproduce the v2 failure analysis

In [30]:
subprocess.run([sys.executable, 'scripts/analyze_v2_for_v3.py'], check=True)


CompletedProcess(args=['/usr/bin/python3', 'scripts/analyze_v2_for_v3.py'], returncode=0)

## 4. Verify or rebuild the v3 dataset
The committed artifact is used by default. Rebuilding is explicit and will refuse to overwrite an existing immutable output directory.

In [31]:
import json

V3_DATA = REPO / 'artifacts/data/v3/train_chat_v3.jsonl'
V3_MANIFEST = REPO / 'artifacts/data/v3/dataset_manifest_v3.json'
assert V3_DATA.exists() and V3_MANIFEST.exists(), 'Committed v3 dataset is missing'
manifest = json.loads(V3_MANIFEST.read_text(encoding='utf-8'))
assert manifest['rows'] == 200
print(json.dumps(manifest, indent=2))


{
  "files": {
    "audit": {
      "path": "artifacts/data/v3/dataset_audit_v3.json",
      "sha256": "88cb2473a5698275fde45e6b608f764ca879e6aa617bbd0bd416fd190cc0ae91"
    },
    "cases": {
      "path": "artifacts/data/v3/train_cases_v3.jsonl",
      "sha256": "5b4efcab413dee38dcb7c855f650c1a838663839c34ba89d7c59f7dc661a97fe"
    },
    "chat": {
      "path": "artifacts/data/v3/train_chat_v3.jsonl",
      "sha256": "c88f499b86b8bc12ba00cfe93c6a1d94afc96565765c397498bc8fe61dd1f63d"
    }
  },
  "rows": 200,
  "settings": {
    "min_long_essay_rate": 0.05,
    "seed": 13,
    "sources": [
      {
        "path": "artifacts/data/train_cases.jsonl",
        "sha256": "cd0d1e1c4aec00955cf1508edeefd49ff4170a3a52541ff5a979f13134832ef6"
      },
      {
        "path": "artifacts/data/v2/train_realistic_v2.jsonl",
        "sha256": "7ad33d7bf9e9fc49f05c3ac9e59861471dc96f6ba24ee633be06d950a17c6886"
      },
      {
        "path": "artifacts/data/v2/train_adversarial_v2.jsonl",
        "sha

## 5. Base 0.5B set1 benchmark

In [32]:
RUN_BASE_BENCHMARK = False  # The verified result is committed; set True to reproduce it.
BASE_EVAL_DIR = RUN_ROOT / 'base'
if RUN_BASE_BENCHMARK:
    subprocess.run([
        sys.executable, 'scripts/eval_v3.py',
        '--model', 'Qwen/Qwen2.5-0.5B-Instruct',
        '--model-name', 'Qwen2.5-0.5B-base',
        '--output-dir', str(BASE_EVAL_DIR),
    ], check=True)
    BASE_SUMMARY = sorted(BASE_EVAL_DIR.glob('*_set1_summary.json'))[-1]
else:
    BASE_SUMMARY = REPO / 'artifacts/eval/v3/base/04b2dada9fdb0cb3e27c_set1_summary.json'
assert BASE_SUMMARY.exists(), BASE_SUMMARY
print(BASE_SUMMARY)


/content/apush-frq-grader-slm/artifacts/eval/v3/base/04b2dada9fdb0cb3e27c_set1_summary.json


## 6. Assemble the three-configuration pretraining benchmark

In [33]:
BENCHMARK_PATH = RUN_ROOT / 'pretraining_dev_benchmark.json'
subprocess.run([
    sys.executable, 'scripts/benchmark_v3_dev.py',
    '--base-summary', str(BASE_SUMMARY),
    '--output', str(BENCHMARK_PATH),
], check=True)
print(BENCHMARK_PATH.read_text(encoding='utf-8'))


{
  "configurations": {
    "base_qwen_0_5b_layered": {
      "count": 27,
      "qwk": 0.2583,
      "repetition_rate": 0.0,
      "schema_valid_rate": 0.7778,
      "total_mae": 2.3704,
      "total_within_one_rate": 0.3704,
      "truncation_count": 1
    },
    "v2_raw_generation": {
      "count": 27,
      "qwk": 0.1825,
      "repetition_rate": 0.2593,
      "schema_valid_rate": 0.3333,
      "total_mae": 3.037,
      "total_within_one_rate": 0.2593,
      "truncation_count": 4
    },
    "v2_with_v3_structured_layer": {
      "count": 27,
      "qwk": 0.2222,
      "repetition_rate": 0.2593,
      "schema_valid_rate": 0.6667,
      "total_mae": 3.3333,
      "total_within_one_rate": 0.2222,
      "truncation_count": 4
    }
  },
  "note": "All three pretraining configurations are populated.",
  "rows": 27,
  "split": "set1"
}


## 7. Verify assistant-only labels and the 3,072-token context

In [34]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_v3_training.py'], check=True)
subprocess.run([
    sys.executable, 'scripts/audit_v3_tokens.py',
    '--output', str(RUN_ROOT / 'v3_training_token_audit.json'),
], check=True)


CompletedProcess(args=['/usr/bin/python3', 'scripts/audit_v3_tokens.py', '--output', '/content/drive/MyDrive/slm_v3/v3_training_token_audit.json'], returncode=0)

## 8. Train Qwen 0.5B
Training is opt-in. Each epoch checkpoint automatically runs set1 generation evaluation.

In [45]:
import torch

RUN_TRAINING = True  # Set True only on a GPU runtime after reviewing the configuration.
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
if RUN_TRAINING:
    assert torch.cuda.is_available(), 'Select a GPU runtime before training'
    DEV05 = (
        f'{sys.executable} scripts/eval_v3.py --model {{checkpoint}} '
        f'--model-name Qwen2.5-0.5B-v3 --output-dir {RUN_ROOT / "dev-0.5b"}'
    )
    command = [
        sys.executable, 'scripts/train_v3.py',
        '--model', 'Qwen/Qwen2.5-0.5B-Instruct',
        '--output', str(RUN_ROOT / 'qwen-0.5b'),
        '--data', str(V3_DATA),
        '--dev-eval-command', DEV05,
    ]
    if not USE_BF16:
        command.append('--no-bf16')
    prior_checkpoints = sorted(
        (RUN_ROOT / 'qwen-0.5b').glob('checkpoint-*'),
        key=lambda path: int(path.name.split('-')[-1]),
    )
    if prior_checkpoints:
        command += ['--resume-from-checkpoint', str(prior_checkpoints[-1])]
        print(f'Resuming from {prior_checkpoints[-1]}')
    run_streaming(command)
else:
    print('0.5B training skipped; set RUN_TRAINING=True to execute.')


Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 583.79it/s]
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.

  0%|          | 0/39 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: to

RuntimeError: Command failed with exit code 1: ['/usr/bin/python3', 'scripts/train_v3.py', '--model', 'Qwen/Qwen2.5-0.5B-Instruct', '--output', '/content/drive/MyDrive/slm_v3/qwen-0.5b', '--data', '/content/apush-frq-grader-slm/artifacts/data/v3/train_chat_v3.jsonl', '--dev-eval-command', '/usr/bin/python3 scripts/eval_v3.py --model {checkpoint} --model-name Qwen2.5-0.5B-v3 --output-dir /content/drive/MyDrive/slm_v3/dev-0.5b']

## 9. Train Qwen 1.5B with identical settings

In [36]:
if RUN_TRAINING:
    DEV15 = (
        f'{sys.executable} scripts/eval_v3.py --model {{checkpoint}} '
        f'--model-name Qwen2.5-1.5B-v3 --output-dir {RUN_ROOT / "dev-1.5b"}'
    )
    command = [
        sys.executable, 'scripts/train_v3.py',
        '--model', 'Qwen/Qwen2.5-1.5B-Instruct',
        '--output', str(RUN_ROOT / 'qwen-1.5b'),
        '--data', str(V3_DATA),
        '--dev-eval-command', DEV15,
    ]
    if not USE_BF16:
        command.append('--no-bf16')
    prior_checkpoints = sorted(
        (RUN_ROOT / 'qwen-1.5b').glob('checkpoint-*'),
        key=lambda path: int(path.name.split('-')[-1]),
    )
    if prior_checkpoints:
        command += ['--resume-from-checkpoint', str(prior_checkpoints[-1])]
        print(f'Resuming from {prior_checkpoints[-1]}')
    run_streaming(command)
else:
    print('1.5B training skipped; set RUN_TRAINING=True to execute.')


1.5B training skipped; set RUN_TRAINING=True to execute.


## 10. Collect checkpoint set1 summaries
The training callback evaluates every saved checkpoint; this cell verifies those summaries exist.

In [37]:
CHECKPOINT_SUMMARIES = sorted((RUN_ROOT / 'dev-0.5b').glob('*_set1_summary.json'))
CHECKPOINT_SUMMARIES += sorted((RUN_ROOT / 'dev-1.5b').glob('*_set1_summary.json'))
print(f'Checkpoint summaries: {len(CHECKPOINT_SUMMARIES)}')
for path in CHECKPOINT_SUMMARIES:
    print(path)


Checkpoint summaries: 0


## 11. Rank passing checkpoints and freeze the candidate

In [38]:
LOCK_MANIFEST = RUN_ROOT / 'v3_final_lock.json'
RANKING_PATH = RUN_ROOT / 'checkpoint_ranking.json'
if CHECKPOINT_SUMMARIES:
    subprocess.run([
        sys.executable, 'scripts/rank_v3_checkpoints.py',
        *map(str, CHECKPOINT_SUMMARIES),
        '--output', str(RANKING_PATH),
        '--lock-manifest', str(LOCK_MANIFEST),
    ], check=True)
else:
    print('No trained checkpoint summaries yet; no candidate was locked.')


No trained checkpoint summaries yet; no candidate was locked.


## 12. Acceptance gate
Set2 remains closed unless layered set1 validity is 100%, truncations are zero, MAE is at most 1.5, within-one is at least 60%, and QWK is at least 0.35.

In [39]:
if LOCK_MANIFEST.exists():
    lock = json.loads(LOCK_MANIFEST.read_text(encoding='utf-8'))
    assert lock.get('set1_acceptance_passed') is True
    print('A passing set1 candidate is locked. Set2 is now eligible for one final run.')
else:
    print('No passing lock manifest. Set2 remains closed.')


No passing lock manifest. Set2 remains closed.


## 13. Official evaluation split lock
Section 13 defaults to set1 and does nothing until explicitly enabled. Set2 additionally requires `FINAL_EVALUATION=True` and the exact passing lock manifest.

In [40]:
RUN_OFFICIAL_EVAL = False
FINAL_EVALUATION = False  # False means set1; True requests the one locked set2 run.
LOCKED_MODEL = None  # Set to the selected checkpoint path from checkpoint_ranking.json.
LOCKED_MODEL_NAME = 'locked-v3-candidate'

if RUN_OFFICIAL_EVAL:
    assert LOCKED_MODEL, 'Set LOCKED_MODEL before evaluation'
    command = [
        sys.executable, 'scripts/eval_v3.py',
        '--model', str(LOCKED_MODEL),
        '--model-name', LOCKED_MODEL_NAME,
        '--output-dir', str(RUN_ROOT / 'official'),
    ]
    if FINAL_EVALUATION:
        assert LOCK_MANIFEST.exists(), 'Set2 requires a passing set1 lock manifest'
        command += ['--final-evaluation', '--lock-manifest', str(LOCK_MANIFEST)]
    subprocess.run(command, check=True)
else:
    print('Official evaluation disabled. Set2 has not been opened.')


Official evaluation disabled. Set2 has not been opened.
